# **XII) ARIMA GARCH Analysis**


1. [**Environment & Config**](#1-env--conf)
2. [**Automated Model Selection**](#2-model--select)
3. [**Performance Benchmarking**](#3-performance)

### **1) Environment & Config** <a id="1-env--conf"></a>


In [1]:
import numpy as np
import pandas as pd
import warnings
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.simplefilter("ignore", ConvergenceWarning)

DATA_PATH = "data/All_Data_Weekly_transformed.csv"
target = "btc_ret"

FOLDS = [
    dict(name="fold1", 
        train=("2014-09-19", "2020-03-27"),
        val=("2020-04-03", "2020-12-25"),    
        test=("2021-01-01", "2021-12-31") 
        ),
    dict(name="fold2",
        train=("2014-09-19", "2021-03-26"),
        val=("2021-04-02", "2021-12-31"),   
        test=("2022-01-07", "2022-12-30") 
    ),
    dict(name="fold3", 
         train=("2014-09-19","2022-03-25"), 
         val=("2022-04-01","2022-12-30"), 
         test=("2023-01-06","2023-12-29")
        ),
    dict(name="fold4", 
         train=("2014-09-19", "2023-03-31"),   
         val=("2023-04-07", "2023-12-29"),     
         test=("2024-01-05", "2025-04-04")    
        ),
]

df = pd.read_csv(DATA_PATH, parse_dates=["date"]).set_index("date").sort_index()
y = df[target].astype(float)

### **2) Automated Model Selection** <a id="2-model--select"></a>


In [2]:
def _clean(arr):
    arr = np.asarray(arr, dtype=float)
    return arr[np.isfinite(arr)]

def select_order_aic(y_train, pmax=6, qmax=6, d=0):
    """
    Fast selection: fit each candidate once on training data and pick the lowest AIC.
    """
    y_train = _clean(y_train)
    best_order, best_aic = None, np.inf

    for p in range(pmax + 1):
        for q in range(qmax + 1):
            if p == 0 and q == 0:
                continue
            order = (p, d, q)
            try:
                res = ARIMA(
                    y_train,
                    order=order,
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                ).fit(method_kwargs={"maxiter": 200})
                if np.isfinite(res.aic) and res.aic < best_aic:
                    best_aic, best_order = res.aic, order
            except Exception:
                pass

    if best_order is None:
        best_order = (1, d, 1)  # safe fallback
    return best_order

def rolling_1step_preds(y_init, y_future, order):
    """
    Fit once on y_init, then 1-step forecasts while appending realized values (no refit).
    """
    y_init = _clean(y_init)
    y_future = _clean(y_future)

    res = ARIMA(
        y_init,
        order=order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(method_kwargs={"maxiter": 200})

    preds = np.empty(len(y_future), dtype=float)
    for i, obs in enumerate(y_future):
        preds[i] = float(res.forecast(1)[0])
        res = res.append([obs], refit=False)
    return preds


### **3) Performance Benchmarking** <a id="3-performance"></a>


In [3]:
# New code combining AR(1), AR(2), and ARIMA
# We also save ARIMA outputs for DM test

import os
os.makedirs("data/arima_outputs", exist_ok=True)

rows = []

for f in FOLDS:
    tr0, tr1 = f["train"]
    va0, va1 = f["val"]
    te0, te1 = f["test"]

    y_tr = y.loc[tr0:tr1].to_numpy()
    y_te = y.loc[te0:te1].dropna().to_numpy()
    y_tv = y.loc[tr0:va1].to_numpy()

    # ARIMA
    order = select_order_aic(y_tr, pmax=3, qmax=3, d=0)
    preds_arima = rolling_1step_preds(y_tv, y_te, order)

    # Save ARIMA predictions for DM test
    np.save(f"data/arima_outputs/{f['name']}_y_true.npy", y_te)
    np.save(f"data/arima_outputs/{f['name']}_y_pred_arima.npy", preds_arima)

    rmse_arima = float(np.sqrt(mean_squared_error(y_te, preds_arima)))
    mae_arima  = float(mean_absolute_error(y_te, preds_arima))
    r2_arima   = float(r2_score(y_te, preds_arima))

    # AR(1)
    preds_ar1 = rolling_1step_preds(y_tv, y_te, order=(1,0,0))

    rmse_ar1 = float(np.sqrt(mean_squared_error(y_te, preds_ar1)))
    mae_ar1  = float(mean_absolute_error(y_te, preds_ar1))
    r2_ar1   = float(r2_score(y_te, preds_ar1))

    # AR(2)
    preds_ar2 = rolling_1step_preds(y_tv, y_te, order=(2,0,0))

    rmse_ar2 = float(np.sqrt(mean_squared_error(y_te, preds_ar2)))
    mae_ar2  = float(mean_absolute_error(y_te, preds_ar2))
    r2_ar2   = float(r2_score(y_te, preds_ar2))

    # Store everything
    rows.append({
        "fold": f["name"],

        "ARIMA_RMSE": rmse_arima,
        "ARIMA_MAE": mae_arima,
        "ARIMA_R2": r2_arima,
        "ARIMA_order": order,

        "AR1_RMSE": rmse_ar1,
        "AR1_MAE": mae_ar1,
        "AR1_R2": r2_ar1,

        "AR2_RMSE": rmse_ar2,
        "AR2_MAE": mae_ar2,
        "AR2_R2": r2_ar2,
    })

    print(f'{f["name"]} | ARIMA RMSE={rmse_arima:.6f} | AR(1) RMSE={rmse_ar1:.6f} | AR(2) RMSE={rmse_ar2:.6f}')

results = pd.DataFrame(rows).set_index("fold").sort_index()
display(results)
print("Mean RMSE:", results["ARIMA_RMSE"].mean())


fold1 | ARIMA RMSE=0.129454 | AR(1) RMSE=0.111463 | AR(2) RMSE=0.113239
fold2 | ARIMA RMSE=0.087573 | AR(1) RMSE=0.086359 | AR(2) RMSE=0.086411
fold3 | ARIMA RMSE=0.067069 | AR(1) RMSE=0.067147 | AR(2) RMSE=0.067477
fold4 | ARIMA RMSE=0.065103 | AR(1) RMSE=0.065154 | AR(2) RMSE=0.065567


,ARIMA_RMSE,ARIMA_MAE,ARIMA_R2,ARIMA_order,AR1_RMSE,AR1_MAE,AR1_R2,AR2_RMSE,AR2_MAE,AR2_R2
fold,,,,,,,,,,
fold1,0.129454,0.106664,-0.341673,"(2, 0, 2)",0.111463,0.087600,0.005340,0.113239,0.089722,-0.026617
fold2,0.087573,0.062283,-0.190383,"(2, 0, 2)",0.086359,0.060618,-0.157593,0.086411,0.060951,-0.159000
fold3,0.067069,0.047679,-0.019639,"(0, 0, 1)",0.067147,0.047609,-0.022008,0.067477,0.047907,-0.032063
fold4,0.065103,0.049696,0.009598,"(0, 0, 1)",0.065154,0.049724,0.008032,0.065567,0.050233,-0.004583


Mean RMSE: 0.08729993140620718


In [4]:
n = len(results)
mean_val = results.mean(numeric_only=True)
std_val = results.std(numeric_only=True)
se_val = std_val / np.sqrt(n)

results.loc["TOTAL (Mean)"] = mean_val
results.loc["TOTAL (SE)"] = se_val

display(results)


,ARIMA_RMSE,ARIMA_MAE,ARIMA_R2,ARIMA_order,AR1_RMSE,AR1_MAE,AR1_R2,AR2_RMSE,AR2_MAE,AR2_R2
fold,,,,,,,,,,
fold1,0.129454,0.106664,-0.341673,"(2, 0, 2)",0.111463,0.087600,0.005340,0.113239,0.089722,-0.026617
fold2,0.087573,0.062283,-0.190383,"(2, 0, 2)",0.086359,0.060618,-0.157593,0.086411,0.060951,-0.159000
fold3,0.067069,0.047679,-0.019639,"(0, 0, 1)",0.067147,0.047609,-0.022008,0.067477,0.047907,-0.032063
fold4,0.065103,0.049696,0.009598,"(0, 0, 1)",0.065154,0.049724,0.008032,0.065567,0.050233,-0.004583
TOTAL (Mean),0.087300,0.066580,-0.135524,NaN,0.082531,0.061388,-0.041557,0.083174,0.062203,-0.055566
TOTAL (SE),0.014942,0.013746,0.081648,NaN,0.010764,0.009190,0.039269,0.011071,0.009603,0.034986
